# Pilot v2.1 — robustness benchmark
This notebook runs a controlled experiment with **public NOAA USCRN Daily01 Tmax (already QC-screened)**. The perturbations, QC candidate flags and reviewer success are **simulated**. A successful reviewer restores the provider-screened reference value (an optimistic oracle). Do not interpret the results as NOAA operational QC findings.


In [ ]:
REPO='https://github.com/priambudihudaf/projects-heatwave-qc-policy-benchmark.git'
BRANCH='research/event-impact-aware-qc-v2-1'
LOCAL='/content/heatwave-qc-v21'
STATIONS=12
REPS=6


In [ ]:
import subprocess, sys, os
from pathlib import Path
repo=Path(LOCAL)
if not repo.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','--depth','1',REPO,str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',BRANCH],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
print('Git commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip())


In [ ]:
# Check implementation and run software-only synthetic demonstration
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],check=True)
subprocess.run([sys.executable,'run_v21.py','--mode','demo','--stations','3','--reps','1','--bootstrap','50'],check=True)


In [ ]:
# Full public NOAA data experiment: injected errors, flags and reviewer outcomes remain synthetic
subprocess.run([sys.executable,'run_v21.py','--mode','real','--stations',str(STATIONS),'--reps',str(REPS)],check=True)


In [ ]:
# Preview primary effects (event-impact against anomaly/hot-run/proximity)
import pandas as pd
folder=repo/'outputs'/'V21_NOAA_CONTROLLED'
effects=pd.read_csv(folder/'v21_paired_effects.csv')
primary=effects[(effects.sample_subset=='all') & (effects.metric=='event_distortion_count') &
                (effects.min_length==3) & (effects.iou_threshold==0.50) &
                (effects.review_budget==0.25) & (effects.review_reliability==0.90)]
display(primary[['detector_regime','fp_placement','comparison','paired_gain','ci95_low','ci95_high']])
print('Positive gain means the event-impact method caused fewer event distortions than comparator.')


In [ ]:
from google.colab import files
import shutil
if not folder.exists(): raise FileNotFoundError('Run the NOAA evaluation first')
archive=shutil.make_archive('/content/heatwave_QC_V21_NOAA_RESULTS','zip',root_dir=folder)
files.download(archive)
